In [1]:
# ==============================================================================
# CELL 1: CÀI ĐẶT MÔI TRƯỜNG & THƯ VIỆN
# ==============================================================================
# Cài đặt FinRL và các thư viện cần thiết
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git -q
!pip install shimmy>=0.2.1 -q
!pip install pandas numpy matplotlib yfinance -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import datetime

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

from stable_baselines3 import PPO, A2C, DDPG
from stable_baselines3.common.logger import configure
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

# Tạo thư mục lưu model
if not os.path.exists("./trained_models"):
    os.makedirs("./trained_models")

print("✅ Đã cài đặt xong thư viện!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/108.7 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Đã cài đặt xong thư viện!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
# ==============================================================================
# CELL 2: ĐỊNH NGHĨA MÔI TRƯỜNG TRAINING (ACTIVE + DIVERSIFICATION)
# ==============================================================================
class StockTradingEnvVietnam_Active(StockTradingEnv):
    def __init__(self,
                 buy_cost_pct=0.001,
                 sell_cost_pct=0.001,
                 sell_tax_pct=0.001,
                 min_trading_lot=100,
                 settlement_cycle=3,
                 stop_loss_pct=-0.07,
                 take_profit_pct=0.15,
                 max_order_pct=0.05,  # Mua tối đa 5% NAV mỗi lệnh
                 max_stock_weight=0.10, # Tỷ trọng tối đa 10% cho 1 mã (ép mua 10 mã)
                 **kwargs):

        if 'num_stock_shares' not in kwargs:
            stock_dim = kwargs.get('stock_dim')
            kwargs['num_stock_shares'] = [0] * stock_dim

        kwargs['buy_cost_pct'] = buy_cost_pct
        kwargs['sell_cost_pct'] = sell_cost_pct

        super().__init__(**kwargs)

        self.sell_tax_pct = sell_tax_pct
        self.min_trading_lot = min_trading_lot
        self.settlement_cycle = settlement_cycle
        self.stop_loss_pct = stop_loss_pct
        self.take_profit_pct = take_profit_pct
        self.max_order_pct = max_order_pct
        self.max_stock_weight = max_stock_weight

        # Quản lý T+
        self.stocks_pending = np.zeros((self.stock_dim, self.settlement_cycle))
        self.cash_pending = np.zeros(self.settlement_cycle)
        self.avg_buy_price = np.zeros(self.stock_dim)

    def reset(self, **kwargs):
        obs, info = super().reset(**kwargs)
        self.stocks_pending = np.zeros((self.stock_dim, self.settlement_cycle))
        self.cash_pending = np.zeros(self.settlement_cycle)
        self.avg_buy_price = np.zeros(self.stock_dim)
        return obs, info

    def _update_settlement(self):
        self.state[0] += self.cash_pending[-1]
        self.cash_pending = np.roll(self.cash_pending, 1); self.cash_pending[0] = 0
        for i in range(self.stock_dim):
            self.state[self.stock_dim + 1 + i] += self.stocks_pending[:, -1][i]
        self.stocks_pending = np.roll(self.stocks_pending, 1, axis=1); self.stocks_pending[:, 0] = 0

    def _get_total_asset_value(self):
        # Tính tổng tài sản hiện tại (NAV)
        # = Tiền mặt + Giá trị CP đang có + Tiền đang về + Giá trị CP đang về

        # 1. Tiền
        cash = self.state[0] + np.sum(self.cash_pending)

        # 2. Cổ phiếu (Khả dụng + Pending)
        current_prices = np.array([self.state[i + 1] for i in range(self.stock_dim)])
        shares_available = np.array([self.state[self.stock_dim + 1 + i] for i in range(self.stock_dim)])
        shares_pending = np.sum(self.stocks_pending, axis=1)

        stock_value = np.sum((shares_available + shares_pending) * current_prices)

        return cash + stock_value

    def _check_stop_loss_take_profit(self):
        current_prices = np.array([self.state[i + 1] for i in range(self.stock_dim)])
        for i in range(self.stock_dim):
            shares = self.state[self.stock_dim + 1 + i]
            if shares > 0 and self.avg_buy_price[i] > 0:
                profit_pct = (current_prices[i] - self.avg_buy_price[i]) / self.avg_buy_price[i]
                if profit_pct <= self.stop_loss_pct or profit_pct >= self.take_profit_pct:
                    revenue = shares * current_prices[i] * (1 - (self.sell_cost_pct + self.sell_tax_pct))
                    self.cash_pending[0] += revenue
                    self.state[self.stock_dim + 1 + i] = 0
                    self.avg_buy_price[i] = 0
                    self.trades += 1

    def step(self, actions):
        self._update_settlement()
        self._check_stop_loss_take_profit()
        return super().step(actions)

    def _sell_stock(self, index, action):
        available = self.state[index + self.stock_dim + 1]
        if available > 0:
            share = min(abs(action) // self.min_trading_lot * self.min_trading_lot, available)
            if share > 0:
                income = self.state[index+1] * share * (1 - (self.sell_cost_pct + self.sell_tax_pct))
                self.cash_pending[0] += income
                self.state[index + self.stock_dim + 1] -= share
                if self.state[index + self.stock_dim + 1] == 0: self.avg_buy_price[index] = 0
                self.trades += 1
                return share
        return 0

    def _buy_stock(self, index, action):
        # --- LOGIC MỚI: KIỂM SOÁT TỶ TRỌNG ---
        total_asset = self._get_total_asset_value()
        current_price = self.state[index+1]

        # 1. Tính giá trị lệnh mua tối đa cho phép (5% NAV)
        max_order_value = total_asset * self.max_order_pct

        # 2. Tính room còn lại cho mã này (Max 10% NAV - Giá trị hiện có)
        shares_holding = self.state[index + self.stock_dim + 1] + np.sum(self.stocks_pending[index])
        current_holding_value = shares_holding * current_price
        max_position_value = total_asset * self.max_stock_weight

        remaining_room = max_position_value - current_holding_value

        # Nếu đã đầy room (giữ > 10%) -> Không cho mua thêm
        if remaining_room <= 0:
            return 0

        # 3. Số tiền được phép mua (Min của các điều kiện)
        # - Không quá số tiền mặt đang có
        # - Không quá 5% NAV (luật lệnh mua)
        # - Không quá room còn lại (luật đa dạng hóa)
        available_cash = self.state[0]
        allowed_cash_to_use = min(available_cash, max_order_value, remaining_room)

        # Chuyển đổi sang số lượng cổ phiếu (lô chẵn)
        # Giá vốn = Giá * (1 + phí)
        cost_per_share = current_price * (1 + self.buy_cost_pct)
        max_shares_can_buy = int(allowed_cash_to_use // cost_per_share)

        # So sánh với số lượng AI muốn mua
        shares_ai_want = action // self.min_trading_lot * self.min_trading_lot

        # Chốt số lượng cuối cùng
        share = min(shares_ai_want, max_shares_can_buy // self.min_trading_lot * self.min_trading_lot)

        if share > 0:
            cost = cost_per_share * share
            self.state[0] -= cost
            self.stocks_pending[index][0] += share

            # Cập nhật giá vốn bình quân
            current_shares = self.state[index + self.stock_dim + 1] + np.sum(self.stocks_pending[index])
            old_val = self.avg_buy_price[index] * (current_shares - share) # share đã cộng vào pending nên phải trừ ra để tính cũ
            # (Logic tính giá vốn đơn giản hóa: Reset lại dựa trên giá mới)
            # Để chính xác:
            prev_shares = current_shares - share
            if prev_shares > 0:
                self.avg_buy_price[index] = (self.avg_buy_price[index] * prev_shares + current_price * share) / current_shares
            else:
                self.avg_buy_price[index] = current_price

            self.trades += 1
            return share
        return 0

print("✅ Đã nâng cấp môi trường: Max 5% lệnh mua & Ép mua ít nhất 10 mã.")

✅ Đã nâng cấp môi trường: Max 5% lệnh mua & Ép mua ít nhất 10 mã.


In [4]:
# ==============================================================================
# CELL 3: LOAD DỮ LIỆU & CẤU HÌNH (ĐÃ FIX LỖI KEYERROR: 0)
# ==============================================================================
import pandas as pd
import os

if os.path.exists("vietnam_vn30_data.csv"):
    df = pd.read_csv("vietnam_vn30_data.csv")
    df['date'] = pd.to_datetime(df['date'])

    # Sắp xếp để đảm bảo thứ tự thời gian
    df = df.sort_values(['date', 'tic']).reset_index(drop=True)

    # 1. Cắt dữ liệu Train TRƯỚC
    train_df = df[(df.date >= '2020-01-01') & (df.date < '2024-01-01')].copy()

    # 2. Xử lý Index cho đúng chuẩn FinRL (BẮT BUỘC)
    # Reset index về 0, 1, 2... thì FinRL mới đọc được ngày 0
    train_df = train_df.sort_values(['date', 'tic']).reset_index(drop=True)

    # Tạo cột index dạng số (để FinRL nhận diện từng ngày)
    # Quan trọng: FinRL dùng index của DataFrame để loop qua từng ngày.
    # Nên index phải là 0, 0, 0 (cho các mã ngày 0), 1, 1, 1 (cho các mã ngày 1)...
    train_df.index = train_df.date.factorize()[0]

    print(f"📊 Dữ liệu Train: {len(train_df)} dòng.")
    print(f"📅 Giai đoạn: {train_df.date.min().date()} -> {train_df.date.max().date()}")

    stock_dimension = len(train_df.tic.unique())
    state_space = 1 + 2*stock_dimension + 4*stock_dimension

    env_kwargs = {
        "hmax": 5000,
        "initial_amount": 1000000000,
        "buy_cost_pct": 0.001,
        "sell_cost_pct": 0.001,
        "sell_tax_pct": 0.001,
        "state_space": state_space,
        "stock_dim": stock_dimension,
        "tech_indicator_list": ['macd', 'rsi_30', 'cci_30', 'dx_30'],
        "action_space": stock_dimension,
        "reward_scaling": 1e-4,
        "stop_loss_pct": -0.07,   # Cắt lỗ -7%
        "take_profit_pct": 0.15,  # Chốt lời +15%
        "max_order_pct": 0.05,    # Mỗi lệnh max 5% NAV
        "max_stock_weight": 0.10  # Mỗi mã max 10% NAV
    }

    e_train_gym = StockTradingEnvVietnam_Active(df=train_df, **env_kwargs)
    print("✅ Môi trường Train đã sẵn sàng (Đã sửa lỗi Index)!")

else:
    print("❌ LỖI: Hãy upload file data.")

📊 Dữ liệu Train: 30000 dòng.
📅 Giai đoạn: 2020-01-02 -> 2023-12-29
✅ Môi trường Train đã sẵn sàng (Đã sửa lỗi Index)!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# ==============================================================================
# CELL 4: HUẤN LUYỆN MODEL PPO (Proximal Policy Optimization)
# ==============================================================================
# PPO thường học tốt hơn trong môi trường có nhiều biến động và quy tắc phức tạp
agent = PPO("MlpPolicy", e_train_gym, verbose=1,
            learning_rate=0.00025,
            n_steps=2048,
            batch_size=64,
            ent_coef=0.01) # Tăng ent_coef để khuyến khích khám phá (tránh lười biếng)

print("🤖 Đang huấn luyện PPO (Active Trader - Kỷ luật cao)...")
# Train 50,000 bước để đủ thời gian học các luật mới
agent.learn(total_timesteps=50000)

# Lưu model
agent.save("trained_models/ppo_vn_agent")
print("✅ Đã lưu model: trained_models/ppo_vn_agent.zip")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🤖 Đang huấn luyện PPO (Active Trader - Kỷ luật cao)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1e+03     |
|    ep_rew_mean     | -2.43e+07 |
| time/              |           |
|    fps             | 127       |
|    iterations      | 1         |
|    time_elapsed    | 16        |
|    total_timesteps | 2048      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1e+03         |
|    ep_rew_mean          | -2.38e+07     |
| time/                   |               |
|    fps                  | 121           |
|    iterations           | 2             |
|    time_elapsed         | 33            |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 8.0316095e-06 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -42.6         |
|    explained_variance   | 8.34e-07      |
|    learning_rate        | 0.00025       |
|    loss                 | 8.44e+10      |
|    n_updates            | 10            |
|    policy_gradient_loss | -0.000348     |
|    std                  | 1             |
|    value_loss           | 1.62e+11      |
-------------------------------------------


...

day: 999, episode: 50
begin_total_asset: 1000000000.00
end_total_asset: 455314575.36
total_reward: -544685424.64
total_cost: 0.00
total_trades: 8518
Sharpe: 0.826


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1e+03        |
|    ep_rew_mean          | -2.42e+07    |
| time/                   |              |
|    fps                  | 110          |
|    iterations           | 25           |
|    time_elapsed         | 463          |
|    total_timesteps      | 51200        |
| train/                  |              |
|    approx_kl            | 5.726266e-06 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -42.6        |
|    explained_variance   | 0            |
|    learning_rate        | 0.00025      |
|    loss                 | 8.18e+10     |
|    n_updates            | 240          |
|    policy_gradient_loss | -0.000287    |
|    std                  | 1            |
|    value_loss           | 1.6e+11      |
------------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Đã lưu model: trained_models/ppo_vn_agent.zip


In [6]:
# ==============================================================================
# CELL 5: HUẤN LUYỆN MODEL DDPG (Deep Deterministic Policy Gradient)
# ==============================================================================
from stable_baselines3.common.noise import NormalActionNoise

# Thêm nhiễu để DDPG chịu khó thử nghiệm các hành động khác nhau
# Giúp nó thoát khỏi bẫy "Buy & Hold"
n_actions = e_train_gym.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

agent_ddpg = DDPG("MlpPolicy", e_train_gym, action_noise=action_noise, verbose=1,
                  learning_rate=0.0005,
                  batch_size=128)

print("🤖 Đang huấn luyện DDPG (Active Trader - Kỷ luật cao)...")
agent_ddpg.learn(total_timesteps=40000)

agent_ddpg.save("trained_models/ddpg_vn_agent")
print("✅ Đã lưu model: trained_models/ddpg_vn_agent.zip")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🤖 Đang huấn luyện DDPG (Active Trader - Kỷ luật cao)...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1e+03     |
|    ep_rew_mean     | -2.71e+06 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 27        |
|    time_elapsed    | 144       |
|    total_timesteps | 4000      |
| train/             |           |
|    actor_loss      | 4.85e+04  |
|    critic_loss     | 3.08e+08  |
|    learning_rate   | 0.0005    |
|    n_updates       | 3899      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 999, episode: 60
begin_total_asset: 1000000000.00
end_total_asset: 1155505313.69
total_reward: 155505313.69
total_cost: 0.00
total_trades: 1224
Sharpe: 0.503
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1e+03     |
|    ep_rew_mean     | -2.51e+06 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 26        |
|    time_elapsed    | 296       |
|    total_timesteps | 8000      |
| train/             |           |
|    actor_loss      | 7.52e+04  |
|    critic_loss     | 9.46e+07  |
|    learning_rate   | 0.0005    |
|    n_updates       | 7899      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1e+03     |
|    ep_rew_mean     | -2.39e+06 |
| time/              |           |
|    episodes        | 12        |
|    fps             | 26        |
|    time_elapsed    | 448       |
|    total_timesteps | 12000     |
| train/             |           |
|    actor_loss      | 7.34e+04  |
|    critic_loss     | 5.58e+07  |
|    learning_rate   | 0.0005    |
|    n_updates       | 11899     |
----------------------------------


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


day: 999, episode: 90
begin_total_asset: 1000000000.00
end_total_asset: 1270136767.41
total_reward: 270136767.41
total_cost: 0.00
total_trades: 1021
Sharpe: 0.502


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1e+03     |
|    ep_rew_mean     | -2.27e+06 |
| time/              |           |
|    episodes        | 40        |
|    fps             | 26        |
|    time_elapsed    | 1524      |
|    total_timesteps | 40000     |
| train/             |           |
|    actor_loss      | 1.52e+05  |
|    critic_loss     | 9.71e+07  |
|    learning_rate   | 0.0005    |
|    n_updates       | 39899     |
----------------------------------
✅ Đã lưu model: trained_models/ddpg_vn_agent.zip


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# ==============================================================================
# CELL 6: HUẤN LUYỆN MODEL A2C (Advantage Actor Critic)
# ==============================================================================
agent_a2c = A2C("MlpPolicy", e_train_gym, verbose=1,
                learning_rate=0.0007,
                ent_coef=0.005)

print("🤖 Đang huấn luyện A2C (Active Trader - Kỷ luật cao)...")
agent_a2c.learn(total_timesteps=40000)

agent_a2c.save("trained_models/a2c_vn_agent")
print("✅ Đã lưu model: trained_models/a2c_vn_agent.zip")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🤖 Đang huấn luyện A2C (Active Trader - Kỷ luật cao)...
-------------------------------------
| time/                 |           |
|    fps                | 122       |
|    iterations         | 100       |
|    time_elapsed       | 4         |
|    total_timesteps    | 500       |
| train/                |           |
|    entropy_loss       | -42.8     |
|    explained_variance | 2.68e-06  |
|    learning_rate      | 0.0007    |
|    n_updates          | 99        |
|    policy_loss        | -3.99e+06 |
|    std                | 1.01      |
|    value_loss         | 1.03e+10  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -2.24e+07 |
| time/                 |           |
|    fps                | 122       |
|    iterations         | 200       |
|    time_elaps

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -2.24e+07 |
| time/                 |           |
|    fps                | 110       |
|    iterations         | 300       |
|    time_elapsed       | 13        |
|    total_timesteps    | 1500      |
| train/                |           |
|    entropy_loss       | -42.9     |
|    explained_variance | 2.38e-07  |
|    learning_rate      | 0.0007    |
|    n_updates          | 299       |
|    policy_loss        | -3.15e+06 |
|    std                | 1.01      |
|    value_loss         | 6.85e+09  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -2.16e+07 |
| time/                 |           |
|    fps                | 112       |
|    iterations         | 400       |
|    time_elapsed       | 17        |
|    total_t

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -2.16e+07 |
| time/                 |           |
|    fps                | 114       |
|    iterations         | 500       |
|    time_elapsed       | 21        |
|    total_timesteps    | 2500      |
| train/                |           |
|    entropy_loss       | -43.1     |
|    explained_variance | 2.38e-07  |
|    learning_rate      | 0.0007    |
|    n_updates          | 499       |
|    policy_loss        | -3.97e+06 |
|    std                | 1.02      |
|    value_loss         | 1.03e+10  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -2.13e+07 |
| time/                 |           |
|    fps                | 109       |
|    iterations         | 600       |
|    time_elapsed       | 27        |
|    total_t

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -6.58e+06 |
| time/                 |           |
|    fps                | 106       |
|    iterations         | 7700      |
|    time_elapsed       | 362       |
|    total_timesteps    | 38500     |
| train/                |           |
|    entropy_loss       | -45.2     |
|    explained_variance | -1.19e-07 |
|    learning_rate      | 0.0007    |
|    n_updates          | 7699      |
|    policy_loss        | -1.25e+05 |
|    std                | 1.09      |
|    value_loss         | 1.04e+07  |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -6.47e+06 |
| time/                 |           |
|    fps                | 106       |
|    iterations         | 7800      |
|    time_elapsed       | 367       |
|    total_t

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -6.47e+06 |
| time/                 |           |
|    fps                | 106       |
|    iterations         | 7900      |
|    time_elapsed       | 372       |
|    total_timesteps    | 39500     |
| train/                |           |
|    entropy_loss       | -45.3     |
|    explained_variance | 5.96e-08  |
|    learning_rate      | 0.0007    |
|    n_updates          | 7899      |
|    policy_loss        | -1.63e+05 |
|    std                | 1.1       |
|    value_loss         | 1.7e+07   |
-------------------------------------
-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 1e+03     |
|    ep_rew_mean        | -6.34e+06 |
| time/                 |           |
|    fps                | 106       |
|    iterations         | 8000      |
|    time_elapsed       | 376       |
|    total_t

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
# ==============================================================================
# CELL 7: TẢI CÁC MODEL VỀ MÁY TÍNH
# ==============================================================================
from google.colab import files

print("⬇️ Đang nén và tải xuống các model...")
!zip -r trained_models_active.zip trained_models/

files.download('trained_models_active.zip')
print("✅ Hoàn tất! Hãy dùng file zip này upload sang File 3 và File 4 để kiểm tra.")

⬇️ Đang nén và tải xuống các model...
  adding: trained_models/ (stored 0%)
  adding: trained_models/ppo_vn_agent.zip (stored 0%)
  adding: trained_models/ddpg_vn_agent.zip (stored 0%)
  adding: trained_models/a2c_vn_agent.zip (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Hoàn tất! Hãy dùng file zip này upload sang File 3 và File 4 để kiểm tra.
